# LoRA: run OK check, smoke run, LR sweep, final runs

One notebook for both LoRA ranks. **Eddie: `RANK = 4`. Dylan: `RANK = 32`** (cell 3). Everything else is the same.

Order (Paddy's 14 Oct list):
1. Setup (T4, Drive, latest code, images into `data/raw`)
2. `python -m src.run` self-check, must end with **run OK**
3. Settings
4. Smoke run, timed. It *is* the first sweep run (lr 1e-3, seed 0), so it isn't wasted
5. LR sweep, validation only: seeds 0–1 first (the minimum), then 2–4 if time allows
6. Summary: pick the lowest mean val CE, flag grid edge / 50-epoch cap, compare seed 2 with frozen, print a decision-log draft
7. Final runs (seeds 0–4) — only after the chosen lr is committed in `configs/r<RANK>.yaml`

Sweep runs go to `runs/pilot/lora_lr/` and `results/pilot_lora_lr_r<RANK>.csv` on Drive, never `runs/` or `runs.csv`.
The test set is never loaded.

**If Colab disconnects:** rerun cells 1 and 3, then the same cell you were on. Finished runs are skipped, a half-done run resumes from `last.pt`.

In [1]:
# 1. Setup (~3-5 min). First: Runtime -> Change runtime type -> T4 GPU
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/aai-project"
REPO = "/content/Advanced-AI"

if not os.path.exists(f"{REPO}/.git"):
    !rm -rf {REPO}
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN").strip()
    !git clone -q https://{TOKEN}@github.com/dylan-teehan-skc/Advanced-AI.git {REPO}
assert os.path.exists(f"{REPO}/.git"), "clone failed: check the GH_TOKEN secret"
%cd {REPO}
!git pull -q
!git log --oneline -1
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

# the self-check and run() read images from data/raw/EuroSAT_RGB inside the repo (Paddy's note)
if not os.path.exists("data/raw/EuroSAT_RGB"):
    !cp "{DRIVE}/EuroSAT_RGB.zip" /content/
    !unzip -q /content/EuroSAT_RGB.zip -d data/raw
!ls data/raw/EuroSAT_RGB
!nvidia-smi --query-gpu=name --format=csv,noheader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Advanced-AI
570df25 (HEAD -> main, origin/main, origin/HEAD) training notebook
AnnualCrop  HerbaceousVegetation  Industrial  PermanentCrop  River
Forest	    Highway		  Pasture     Residential    SeaLake
Tesla T4


In [4]:
# 2. Self-check (~2-4 min, CPU, downloads the ViT once). Last line must be: run OK
!python -m src.run

Loading weights: 100% 198/198 [00:00<00:00, 3852.89it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
rank 4: 155,146 trainable parameters (head 7,690 + LoRA 147,456)
epoch 1: train CE 2.3338, val CE 2.2872  (best)
epoch 2: train CE 2.2858, val CE 2.2514  (best)
r4_s0 already finished, skipping
{'run_id': 'r4_s0', 'condition': 'r4', 'rank': '4', 'seed': '0', 'n_per_class': '2', 'lr': '0.001', 'alpha': '4', 'best_epoch': '2', 'val_ce': '2.251406', 'val_bal_acc': '0.15', 'test_ce': '',

In [2]:
# 3. Settings. Rerun this after any disconnect.
import time
from pathlib import Path

import pandas as pd
import torch
import yaml

from src.run import load_config, run

RANK = 4                      # Eddie 4, Dylan 32
COND = f"r{RANK}"
LRS = [1e-4, 3e-4, 1e-3]      # Paddy's grid
SWEEP_ROOT = Path(DRIVE) / "runs/pilot/lora_lr"
SWEEP_CSV = f"{DRIVE}/results/pilot_lora_lr_{COND}.csv"
FINAL_CSV = f"{DRIVE}/results/runs_{COND}.csv"   # own CSV per rank, so two accounts never append to one file

base = load_config(f"configs/{COND}.yaml")
assert base["n_per_class"] == 20, "old config: git pull didn't bring in Paddy's N = 20 commit"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")


def lr_folder(lr):
    return SWEEP_ROOT / f"{COND}_lr{lr:g}"


def sweep_config(lr):
    """configs/r<RANK>.yaml with only lr changed, in its own folder so runs at different lrs never collide."""
    folder = lr_folder(lr)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "config.yaml").write_text(yaml.safe_dump({**base, "lr": lr}))
    return folder / "config.yaml", folder


def epochs_run(folder, seed):
    """How many epochs the run actually trained (early stop or the 50 cap), from its last checkpoint."""
    return torch.load(Path(folder) / f"{COND}_s{seed}" / "last.pt", map_location="cpu")["state"]["epoch"]


# a leftover folder in runs/ would be skipped (row.json) or resumed (last.pt) instead of run fresh
stale = sorted(p.name for p in (Path(DRIVE) / "runs").glob(f"{COND}_s*"))
print(f"Already in runs/: {stale}. Fine if they are your final runs resuming; otherwise move them out before cell 7"
      if stale else "runs/ is clear for the final runs")
old = sorted(str(p.relative_to(SWEEP_ROOT)) for p in SWEEP_ROOT.glob(f"{COND}_lr*/{COND}_s*"))
print(f"sweep runs already on Drive (will be skipped or resumed): {old}" if old else "no sweep runs yet")

GPU: Tesla T4
runs/ is clear for the final runs
sweep runs already on Drive (will be skipped or resumed): ['r4_lr0.0001/r4_s0', 'r4_lr0.0001/r4_s1', 'r4_lr0.0001/r4_s2', 'r4_lr0.0003/r4_s0', 'r4_lr0.0003/r4_s1', 'r4_lr0.0003/r4_s2', 'r4_lr0.001/r4_s0', 'r4_lr0.001/r4_s1', 'r4_lr0.001/r4_s2']


In [6]:
# 4. Smoke run on split_v1, timed. Same settings as sweep run lr 1e-3 / seed 0, so it counts towards the sweep.
cfg, folder = sweep_config(1e-3)
row = run(cfg, seed=0, out_root=folder, results_csv=SWEEP_CSV)
ep = epochs_run(folder, 0)
mins = row["train_time_s"] / 60
print(f"\nsmoke run: {mins:.1f} min training, {ep} epochs ({row['train_time_s'] / ep:.0f} s/epoch), "
      f"best epoch {row['best_epoch']}, val CE {row['val_ce']:.4f}, balanced acc {row['val_bal_acc']:.3f}, "
      f"{row['trainable_params']:,} trainable")
print(f"rough budget if every run is like this one (+~1 min each to load and score):\n"
      f"  full sweep (14 more) + 5 finals: {19 * (mins + 1) / 60:.1f} h\n"
      f"  seeds 0-1 sweep (5 more) + 5 finals: {10 * (mins + 1) / 60:.1f} h")
print("Lower lrs usually run more epochs, so treat this as a floor.")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 4: 155,146 trainable parameters (head 7,690 + LoRA 147,456)
epoch 1: train CE 2.2733, val CE 2.1657  (best)
epoch 2: train CE 2.0828, val CE 2.0090  (best)
epoch 3: train CE 1.8770, val CE 1.8352  (best)
epoch 4: train CE 1.6765, val CE 1.6417  (best)
epoch 5: train CE 1.4509, val CE 1.4368  (best)
epoch 6: train CE 1.2190, val CE 1.2421  (best)
epoch 7: train CE 1.0156, val CE 1.0767  (best)
epoch 8: train CE 0.8386, val CE 0.9066  (best)
epoch 9: train CE 0.6390, val CE 0.7716  (best)
epoch 10: train CE 0.4916, val CE 0.6520  (best)
epoch 11: train CE 0.3572, val CE 0.5759  (best)
epoch 12: train CE 0.2732, val CE 0.5138  (best)
epoch 13: train CE 0.2079, val CE 0.4635  (best)
epoch 14: train CE 0.1557, val CE 0.4115  (best)
epoch 15: train CE 0.1223, val CE 0.3999  (best)
epoch 16: train CE 0.1017, val CE 0.3801  (best)
epoch 17: train CE 0.0824, val CE 0.4337
epoch 18: train CE 0.0744, val CE 0.3937
epoch 19: train CE 0.0565, val CE 0.3922
epoch 20: train CE 0.0555, val CE 0.3

In [3]:
# 5a. Sweep, seeds 0-1 for every lr (the minimum). One seed at a time across all lrs,
#     so if Colab cuts us off, every lr has the same seeds.
def sweep(seeds):
    for seed in seeds:
        for lr in LRS:
            cfg, folder = sweep_config(lr)
            r = run(cfg, seed=seed, out_root=folder, results_csv=SWEEP_CSV)
            print(f"== lr {lr:g} seed {seed}: val CE {r['val_ce']:.4f}, balanced acc {r['val_bal_acc']:.3f}, "
                  f"best epoch {r['best_epoch']} of {epochs_run(folder, seed)}, {r['train_time_s'] / 60:.1f} min\n")


sweep([0, 1])

r4_s0 already finished, skipping
== lr 0.0001 seed 0: val CE 1.1440, balanced acc 0.798, best epoch 50 of 50, 42.1 min

r4_s0 already finished, skipping
== lr 0.0003 seed 0: val CE 0.3853, balanced acc 0.888, best epoch 50 of 50, 42.1 min

r4_s0 already finished, skipping
== lr 0.001 seed 0: val CE 0.3432, balanced acc 0.885, best epoch 25 of 30, 24.9 min

r4_s1 already finished, skipping
== lr 0.0001 seed 1: val CE 1.1412, balanced acc 0.784, best epoch 50 of 50, 42.1 min

r4_s1 already finished, skipping
== lr 0.0003 seed 1: val CE 0.4074, balanced acc 0.862, best epoch 50 of 50, 42.1 min

r4_s1 already finished, skipping
== lr 0.001 seed 1: val CE 0.3290, balanced acc 0.882, best epoch 30 of 35, 29.5 min



In [4]:
# 5b. Seeds 2-4, to match frozen's 5 seeds per value. Skip if time is short, and say so in the log.
sweep([2, 3, 4])

r4_s2 already finished, skipping
== lr 0.0001 seed 2: val CE 1.1674, balanced acc 0.765, best epoch 50 of 50, 42.1 min

r4_s2 already finished, skipping
== lr 0.0003 seed 2: val CE 0.4015, balanced acc 0.871, best epoch 50 of 50, 42.1 min



config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 4: 155,146 trainable parameters (head 7,690 + LoRA 147,456)
epoch 1: train CE 2.2993, val CE 2.2662  (best)
epoch 2: train CE 2.2374, val CE 2.2203  (best)
epoch 3: train CE 2.1842, val CE 2.1758  (best)
epoch 4: train CE 2.1284, val CE 2.1289  (best)
epoch 5: train CE 2.0669, val CE 2.0785  (best)
epoch 6: train CE 2.0149, val CE 2.0216  (best)
epoch 7: train CE 1.9334, val CE 1.9583  (best)
epoch 8: train CE 1.8659, val CE 1.8906  (best)
epoch 9: train CE 1.7805, val CE 1.8209  (best)
epoch 10: train CE 1.7019, val CE 1.7453  (best)
epoch 11: train CE 1.6113, val CE 1.6708  (best)
epoch 12: train CE 1.5241, val CE 1.5947  (best)
epoch 13: train CE 1.4304, val CE 1.5165  (best)
epoch 14: train CE 1.3436, val CE 1.4328  (best)
epoch 15: train CE 1.2577, val CE 1.3470  (best)
epoch 16: train CE 1.1491, val CE 1.2691  (best)
epoch 17: train CE 1.0722, val CE 1.1964  (best)
epoch 18: train CE 0.9825, val CE 1.1217  (best)
epoch 19: train CE 0.8859, val CE 1.0537  (best)
epoch 20: tra

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 4: 155,146 trainable parameters (head 7,690 + LoRA 147,456)
epoch 1: train CE 2.2615, val CE 2.1566  (best)
epoch 2: train CE 2.0668, val CE 1.9982  (best)
epoch 3: train CE 1.8737, val CE 1.8221  (best)
epoch 4: train CE 1.6631, val CE 1.6263  (best)
epoch 5: train CE 1.4375, val CE 1.4144  (best)
epoch 6: train CE 1.2050, val CE 1.2282  (best)
epoch 7: train CE 0.9829, val CE 1.0448  (best)
epoch 8: train CE 0.7801, val CE 0.8747  (best)
epoch 9: train CE 0.6005, val CE 0.7438  (best)
epoch 10: train CE 0.4545, val CE 0.6352  (best)
epoch 11: train CE 0.3432, val CE 0.5303  (best)
epoch 12: train CE 0.2596, val CE 0.4878  (best)
epoch 13: train CE 0.1902, val CE 0.4369  (best)
epoch 14: train CE 0.1504, val CE 0.3957  (best)
epoch 15: train CE 0.1172, val CE 0.4684
epoch 16: train CE 0.0910, val CE 0.3760  (best)
epoch 17: train CE 0.0783, val CE 0.3726  (best)
epoch 18: train CE 0.0596, val CE 0.3497  (best)
epoch 19: train CE 0.0496, val CE 0.3423  (best)
epoch 20: train CE 0.

In [5]:
# 6. Summary and pick. Rule: lowest mean validation CE, over the seeds every lr has finished.
df = pd.read_csv(SWEEP_CSV)
df = df[df["condition"] == COND].drop_duplicates(["lr", "seed"], keep="last")
seeds = sorted(set.intersection(*(set(g["seed"]) for _, g in df.groupby("lr"))))
missing = set(LRS) - set(df["lr"])
assert not missing, f"no finished runs yet for lr {missing}"
df = df[df["seed"].isin(seeds)].copy()
df["epochs"] = [epochs_run(lr_folder(lr), s) for lr, s in zip(df["lr"], df["seed"])]

t = df.groupby("lr").agg(runs=("seed", "count"), ce=("val_ce", "mean"), ce_sd=("val_ce", "std"),
                         acc=("val_bal_acc", "mean"), acc_sd=("val_bal_acc", "std"),
                         best_epochs=("best_epoch", list), epochs=("epochs", list),
                         mins=("train_time_s", lambda s: round(s.mean() / 60, 1))).reset_index()
print(f"{COND}, N = 20, seeds {seeds}\n")
print(t.round(4).to_string(index=False))

best = t.loc[t["ce"].idxmin()]
CHOSEN_LR = float(best["lr"])
print(f"\nRule: lowest mean val CE -> lr = {CHOSEN_LR:g}")
flags = []
if CHOSEN_LR in (min(LRS), max(LRS)):
    flags.append(f"the best lr ({CHOSEN_LR:g}) is at the edge of the grid; the true best may lie outside it")
cap = base["max_epochs"]
hit = df[df["epochs"] >= cap]
if len(hit):
    flags.append("hit the 50-epoch cap: " + ", ".join(f"lr {r.lr:g} s{r.seed} (best epoch {r.best_epoch})"
                                                       for r in hit.itertuples()))
if len(seeds) < 5:
    flags.append(f"only seeds {seeds} for every lr (frozen used 5)")
for f in flags:
    print("FLAG:", f)

# seed 2 was frozen's weak seed (0.605); is it weak for LoRA too?
frozen = pd.read_csv("results/runs.csv").query("condition == 'frozen'").set_index("seed")["val_ce"]
mine = df[df["lr"] == CHOSEN_LR].set_index("seed")["val_ce"]
print("\nval CE per seed, frozen vs this rank at the chosen lr:")
print(pd.DataFrame({"frozen": frozen, f"{COND} lr {CHOSEN_LR:g}": mine}).round(3).to_string())

# draft decision-log entry (Paddy's format). Check it, edit the Why if needed, then paste into your log.
per_lr = "; ".join(f"lr {r.lr:g} gave CE {r.ce:.3f} +/- {r.ce_sd:.3f}" for r in t.itertuples())
print(f"""
---- draft for docs/decision-log ----
## {time.strftime('%-d %b %Y')} - {COND} learning rate = {CHOSEN_LR:g}
- **Decision:** {COND} uses lr = {CHOSEN_LR:g}, chosen by the sweep rule: lowest mean validation CE at N = 20 over 1e-4, 3e-4, 1e-3.
- **Why:** {len(seeds)} seeds each: {per_lr}.{' ' + '. '.join(f[0].upper() + f[1:] for f in flags) + '.' if flags else ' The best is inside the grid and no run hit the 50-epoch cap.'} Results in `results/pilot_lora_lr_{COND}.csv`.
- **Affects:** `lr` in `configs/{COND}.yaml`, the final {COND} runs.""")

r4, N = 20, seeds [0, 1, 2, 3, 4]

    lr  runs     ce  ce_sd    acc  acc_sd          best_epochs               epochs  mins
0.0001     5 1.1410 0.0352 0.8001  0.0329 [50, 50, 50, 50, 50] [50, 50, 50, 50, 50]  42.5
0.0003     5 0.3831 0.0254 0.8820  0.0150 [50, 50, 50, 50, 48] [50, 50, 50, 50, 50]  42.5
0.0010     5 0.3356 0.0192 0.8867  0.0079 [25, 30, 24, 27, 21] [30, 35, 29, 32, 26]  25.8

Rule: lowest mean val CE -> lr = 0.001
FLAG: the best lr (0.001) is at the edge of the grid; the true best may lie outside it
FLAG: hit the 50-epoch cap: lr 0.0001 s0 (best epoch 50), lr 0.0003 s0 (best epoch 50), lr 0.0001 s1 (best epoch 50), lr 0.0003 s1 (best epoch 50), lr 0.0001 s2 (best epoch 50), lr 0.0003 s2 (best epoch 50), lr 0.0001 s3 (best epoch 50), lr 0.0003 s3 (best epoch 50), lr 0.0001 s4 (best epoch 50), lr 0.0003 s4 (best epoch 48)

val CE per seed, frozen vs this rank at the chosen lr:
      frozen  r4 lr 0.001
seed                     
0      0.547        0.343
1      0.550     

### Before cell 7 (on your laptop, ~5 min)
1. `git pull`
2. In `configs/r<RANK>.yaml` set `lr:` to the chosen value **written with a dot and a signed exponent**, e.g. `3.0e-4`. PyYAML reads `3e-4` as a *string*, not a number.
3. Paste the decision-log draft into your log, copy `pilot_lora_lr_r<RANK>.csv` from Drive `aai-project/results/` into the repo's `results/`.
4. Commit and push. Then run cell 7 (it pulls first, so each row records the commit with the chosen lr).

In [6]:
# 7. Final runs: seeds 0-4 at the chosen lr into runs/ and runs_r<RANK>.csv. Validation only.
!git pull -q
!git log --oneline -1
CHOSEN_LR = globals().get("CHOSEN_LR")  # after a restart, type it instead, e.g. CHOSEN_LR = 3e-4
assert CHOSEN_LR is not None, "set CHOSEN_LR (cell 6 sets it, or type it)"
cfg = load_config(f"configs/{COND}.yaml")
assert isinstance(cfg["lr"], float), f"lr in configs/{COND}.yaml read as {cfg['lr']!r}; write it like 3.0e-4"
assert abs(cfg["lr"] - CHOSEN_LR) < 1e-12, f"configs/{COND}.yaml still has lr {cfg['lr']:g}; commit the chosen lr and push first"

for seed in range(5):
    r = run(f"configs/{COND}.yaml", seed=seed, out_root=f"{DRIVE}/runs", results_csv=FINAL_CSV)
    print(f"== {r['run_id']}: val CE {r['val_ce']:.4f}, balanced acc {r['val_bal_acc']:.3f}, best epoch {r['best_epoch']}\n")

final = pd.read_csv(FINAL_CSV)
print(final[["run_id", "lr", "best_epoch", "val_ce", "val_bal_acc", "train_time_s", "gpu", "git_commit"]].to_string(index=False))
print(f"\nmean val CE {final['val_ce'].mean():.3f} +/- {final['val_ce'].std():.3f}, "
      f"balanced acc {final['val_bal_acc'].mean():.1%}   (frozen: 0.559 +/- 0.026, 80.2%)")

570df25 (HEAD -> main, origin/main, origin/HEAD) training notebook


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 4: 155,146 trainable parameters (head 7,690 + LoRA 147,456)
epoch 1: train CE 2.2733, val CE 2.1657  (best)
epoch 2: train CE 2.0828, val CE 2.0090  (best)
epoch 3: train CE 1.8770, val CE 1.8352  (best)
epoch 4: train CE 1.6765, val CE 1.6417  (best)
epoch 5: train CE 1.4509, val CE 1.4367  (best)
epoch 6: train CE 1.2188, val CE 1.2421  (best)
epoch 7: train CE 1.0159, val CE 1.0780  (best)
epoch 8: train CE 0.8398, val CE 0.9081  (best)
epoch 9: train CE 0.6401, val CE 0.7736  (best)
epoch 10: train CE 0.4922, val CE 0.6541  (best)
epoch 11: train CE 0.3588, val CE 0.5789  (best)
epoch 12: train CE 0.2754, val CE 0.5168  (best)
epoch 13: train CE 0.2079, val CE 0.4639  (best)
epoch 14: train CE 0.1588, val CE 0.4149  (best)
epoch 15: train CE 0.1221, val CE 0.4054  (best)
epoch 16: train CE 0.1029, val CE 0.3778  (best)
epoch 17: train CE 0.0850, val CE 0.4313
epoch 18: train CE 0.0768, val CE 0.3969
epoch 19: train CE 0.0574, val CE 0.3989
epoch 20: train CE 0.0567, val CE 0.3

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 4: 155,146 trainable parameters (head 7,690 + LoRA 147,456)
epoch 1: train CE 2.2630, val CE 2.1754  (best)
epoch 2: train CE 2.0705, val CE 2.0146  (best)
epoch 3: train CE 1.8838, val CE 1.8413  (best)
epoch 4: train CE 1.6811, val CE 1.6429  (best)
epoch 5: train CE 1.4639, val CE 1.4530  (best)
epoch 6: train CE 1.2660, val CE 1.2663  (best)
epoch 7: train CE 1.0362, val CE 1.0935  (best)
epoch 8: train CE 0.8643, val CE 0.9725  (best)
epoch 9: train CE 0.6990, val CE 0.8307  (best)
epoch 10: train CE 0.5622, val CE 0.7647  (best)
epoch 11: train CE 0.4300, val CE 0.6224  (best)
epoch 12: train CE 0.3385, val CE 0.5788  (best)
epoch 13: train CE 0.2459, val CE 0.4865  (best)
epoch 14: train CE 0.1907, val CE 0.4463  (best)
epoch 15: train CE 0.1411, val CE 0.3986  (best)
epoch 16: train CE 0.1026, val CE 0.3725  (best)
epoch 17: train CE 0.0821, val CE 0.3487  (best)
epoch 18: train CE 0.0672, val CE 0.3559
epoch 19: train CE 0.0577, val CE 0.3520
epoch 20: train CE 0.0530, va

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 4: 155,146 trainable parameters (head 7,690 + LoRA 147,456)
epoch 1: train CE 2.2615, val CE 2.1566  (best)
epoch 2: train CE 2.0668, val CE 1.9982  (best)
epoch 3: train CE 1.8737, val CE 1.8221  (best)
epoch 4: train CE 1.6631, val CE 1.6263  (best)
epoch 5: train CE 1.4381, val CE 1.4148  (best)
epoch 6: train CE 1.2052, val CE 1.2293  (best)
epoch 7: train CE 0.9839, val CE 1.0477  (best)
epoch 8: train CE 0.7806, val CE 0.8760  (best)
epoch 9: train CE 0.6017, val CE 0.7468  (best)
epoch 10: train CE 0.4558, val CE 0.6389  (best)
epoch 11: train CE 0.3449, val CE 0.5339  (best)
epoch 12: train CE 0.2587, val CE 0.4874  (best)
epoch 13: train CE 0.1911, val CE 0.4383  (best)
epoch 14: train CE 0.1505, val CE 0.3958  (best)
epoch 15: train CE 0.1185, val CE 0.4759
epoch 16: train CE 0.0925, val CE 0.3818  (best)
epoch 17: train CE 0.0779, val CE 0.3725  (best)
epoch 18: train CE 0.0596, val CE 0.3571  (best)
epoch 19: train CE 0.0501, val CE 0.3481  (best)
epoch 20: train CE 0.